In [1]:
import pandas as pd
import numpy as np
from time import perf_counter
from datasets import load_dataset
from memory_profiler import memory_usage
from tqdm import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import BertTokenizer, BertModel
from sklearn.preprocessing import MultiLabelBinarizer


In [2]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

BATCH_SIZE = 16
LEARNING_RATE = 3e-5

PRETRAINED_MODEL_NAME = 'bert-base-uncased'
MAX_LEN = 128
MAX_EPOCHS = 4  # Maximum epochs for early stopping
PATIENCE = 3     # Patience for early stopping

tokenizer = BertTokenizer.from_pretrained(PRETRAINED_MODEL_NAME)

print(f"Using device: {DEVICE}")

Using device: cuda


In [3]:
ds = load_dataset("TimSchopf/arxiv_categories", "default")

train_df = ds['train'].to_pandas()
val_df = ds['validation'].to_pandas()
test_df = ds['test'].to_pandas()

train_df

,id,title,abstract,categories,creation_date
0,2204.14117,A Comparative Study of Meter Detection Methods...,In order to read meter values from a camera on...,[Computer Science Archive->cs.CV],2022-04-24 13:59:57+00:00
1,2305.19887,The Markov chain embedding problem in a low ju...,We consider the problem of finding the transit...,[Mathematics Archive->math.PR],2023-05-31 14:24:25+00:00
2,0910.5857,Chaotic Transport and Chronology of Complex As...,We present a transport model that describes th...,[Physics Archive->astro-ph->astro-ph.EP],2009-10-30 12:34:26+00:00
3,1801.10207,FITing-Tree: A Data-aware Index Structure,Index structures are one of the most important...,[Computer Science Archive->cs.DB],2018-01-30 20:22:53+00:00
4,0803.0849,The Universal Cardinal Ordering of Fixed Points,"We present the theorem which determines, by a ...",[Physics Archive->nlin->nlin.CD],2008-03-06 12:55:48+00:00
...,...,...,...,...,...
163163,1805.11049,Induced Chern-Simons modified gravity at finit...,We calculate the linearized four-dimensional g...,"[Physics Archive->gr-qc, Physics Archive->hep-...",2018-05-28 17:00:59+00:00
163164,1907.11966,Small Time Behavior and Summability for the Sc...,We consider the Carleson's problem regarding s...,[Mathematics Archive->math.AP],2019-07-27 18:59:04+00:00
163165,1510.08071,GM2Calc: Precise MSSM prediction for $(g - 2)$...,"We present GM2Calc, a public C++ program for t...",[Physics Archive->hep->hep-ph],2015-10-27 20:09:29+00:00
163166,1803.01475,"The Fu-Yau equation on compact astheno-K\""ahle...","In this paper, we study the Fu-Yau equation on...","[Mathematics Archive->math.AP, Mathematics Arc...",2018-03-05 02:54:16+00:00


In [4]:
train_df = train_df.rename(columns={'title': 'text'})
val_df = val_df.rename(columns={'title': 'text'})
test_df = test_df.rename(columns={'title': 'text'})

train_df = train_df.rename(columns={'categories': 'labels'})
val_df = val_df.rename(columns={'categories': 'labels'})
test_df = test_df.rename(columns={'categories': 'labels'})

In [5]:
allowed_categories = ["cs.AI", "cs.CL", "stat.ML", "math.OC", "cs.LG"]

def clean_element(lst):
    final = []
    for elem in lst:
        clean = elem.split('->')[-1]
        final.append(clean)
    return final

train_df['labels'] = train_df['labels'].apply(clean_element)
val_df['labels'] = val_df['labels'].apply(clean_element)
test_df['labels'] = test_df['labels'].apply(clean_element)

In [6]:
train_df = train_df[train_df['labels'].apply(lambda cats: all(c in allowed_categories for c in cats))]
test_df = test_df[test_df['labels'].apply(lambda cats: all(c in allowed_categories for c in cats))]
val_df = val_df[val_df['labels'].apply(lambda cats: all(c in allowed_categories for c in cats))]

train_df = train_df[train_df['labels'].apply(len) > 0]
test_df = test_df[test_df['labels'].apply(len) > 0]
val_df = val_df[val_df['labels'].apply(len) > 0]

In [7]:
train_df.drop(columns=['id','abstract','creation_date'], inplace=True)
test_df.drop(columns=['id','abstract','creation_date'], inplace=True)
val_df.drop(columns=['id','abstract','creation_date'], inplace=True)

train_df.reset_index(drop=True, inplace=True)
test_df.reset_index(drop=True, inplace=True)
val_df.reset_index(drop=True, inplace=True)

train_df

,text,labels
0,Upper and Lower Bounds for Large Scale Multist...,[math.OC]
1,Binary Classification: Counterbalancing Class ...,[cs.LG]
2,Smooth Optimization with Approximate Gradient,[math.OC]
3,An AI-powered Smart Routing Solution for Payme...,[cs.AI]
4,A linearly convergent method for solving high-...,[math.OC]
...,...,...
10141,Simple Question Answering with Subgraph Rankin...,"[cs.CL, cs.LG, stat.ML]"
10142,"Fire Now, Fire Later: Alarm-Based Systems for ...","[cs.AI, cs.LG, stat.ML]"
10143,NSP-BERT: A Prompt-based Few-Shot Learner Thro...,"[cs.AI, cs.CL]"
10144,Near-optimal bounds for phase synchronization,[math.OC]


In [8]:
mlb = MultiLabelBinarizer()

train_labels_binarized = mlb.fit_transform(train_df['labels'])
val_labels_binarized = mlb.transform(val_df['labels'])
test_labels_binarized = mlb.transform(test_df['labels'])

train_labels_df = pd.DataFrame(train_labels_binarized, columns=mlb.classes_)
val_labels_df = pd.DataFrame(val_labels_binarized, columns=mlb.classes_)
test_labels_df = pd.DataFrame(test_labels_binarized, columns=mlb.classes_)

train_df = pd.concat([train_df, train_labels_df], axis=1)
val_df = pd.concat([val_df, val_labels_df], axis=1)
test_df = pd.concat([test_df, test_labels_df], axis=1)

train_df = train_df.drop(columns=['labels'])
val_df = val_df.drop(columns=['labels'])
test_df = test_df.drop(columns=['labels'])

train_df

,text,cs.AI,cs.CL,cs.LG,math.OC,stat.ML
0,Upper and Lower Bounds for Large Scale Multist...,0,0,0,1,0
1,Binary Classification: Counterbalancing Class ...,0,0,1,0,0
2,Smooth Optimization with Approximate Gradient,0,0,0,1,0
3,An AI-powered Smart Routing Solution for Payme...,1,0,0,0,0
4,A linearly convergent method for solving high-...,0,0,0,1,0
...,...,...,...,...,...,...
10141,Simple Question Answering with Subgraph Rankin...,0,1,1,0,1
10142,"Fire Now, Fire Later: Alarm-Based Systems for ...",1,0,1,0,1
10143,NSP-BERT: A Prompt-based Few-Shot Learner Thro...,1,1,0,0,0
10144,Near-optimal bounds for phase synchronization,0,0,0,1,0


In [11]:
class MultiLabelClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        """
        Args:
            texts: List or array of text samples
            labels: 2D array of shape (num_samples, num_classes) with binary indicators (0 or 1)
            tokenizer: Pretrained tokenizer (e.g., BertTokenizer)
            max_len: Maximum sequence length
        """
        self.texts = texts
        self.labels = labels  # Shape: (num_samples, num_classes)
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]  # Shape: (num_classes,)
        
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.float)  # Binary vector for multilabel
        }

In [12]:
class BertForMultiLabelClassification(nn.Module):
    def __init__(self, num_classes):
        super(BertForMultiLabelClassification, self).__init__()
        self.bert = BertModel.from_pretrained(PRETRAINED_MODEL_NAME)
        self.pre_classifier = nn.Linear(768, 768)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(768, num_classes)  # Output logits for each class
    
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        hidden_state = outputs[0][:, 0]  # CLS token
        pooled_output = self.pre_classifier(hidden_state)
        pooled_output = nn.ReLU()(pooled_output)
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits  # Return raw logits for BCEWithLogitsLoss

In [13]:
def get_metrics(y_true, y_pred):

    acc = accuracy_score(y_true, y_pred)

    precisions, recalls, f1s, supports = precision_recall_fscore_support(y_true, y_pred)

    return acc, precisions, recalls, f1s

In [14]:
def train_model(model, train_dataloader, val_dataloader, optimizer, criterion, save_path, max_epochs=MAX_EPOCHS, patience=PATIENCE):
    best_val_loss = float('inf')
    epochs_no_improve = 0
    start_train = perf_counter()
    
    # Initialize best metrics
    best_train_acc = 0
    best_train_precisions = None
    best_train_recalls = None
    best_train_f1s = None
    best_val_acc = 0
    best_val_precisions = None
    best_val_recalls = None
    best_val_f1s = None
    
    for epoch in range(max_epochs):
        model.train()
        train_loss = 0
        train_preds = []
        train_true = []
        
        for batch in tqdm(train_dataloader, desc=f'Epoch {epoch + 1}/{max_epochs}', leave=False):
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)  # Shape: (batch_size, num_classes)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)  # Shape: (batch_size, num_classes)
            loss = criterion(outputs, labels)  # BCEWithLogitsLoss
            train_loss += loss.item()
            # Compute binary predictions for each class
            preds = (torch.sigmoid(outputs) > 0.5).float().cpu().numpy()  # Shape: (batch_size, num_classes)
            train_preds.extend(preds)
            train_true.extend(labels.cpu().numpy())
            loss.backward()
            optimizer.step()
        
        train_loss /= len(train_dataloader)
        train_true = np.array(train_true)  # Shape: (num_samples, num_classes)
        train_preds = np.array(train_preds)  # Shape: (num_samples, num_classes)
        train_acc, train_precisions, train_recalls, train_f1s = get_metrics(train_true, train_preds)
        
        model.eval()
        val_loss = 0
        val_preds = []
        val_true = []
        with torch.no_grad():
            for batch in val_dataloader:
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                labels = batch['labels'].to(DEVICE)
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                preds = (torch.sigmoid(outputs) > 0.5).float().cpu().numpy()
                val_preds.extend(preds)
                val_true.extend(labels.cpu().numpy())
        
        val_loss /= len(val_dataloader)
        val_true = np.array(val_true)  # Shape: (num_samples, num_classes)
        val_preds = np.array(val_preds)  # Shape: (num_samples, num_classes)
        val_acc, val_precisions, val_recalls, val_f1s = get_metrics(val_true, val_preds)
        
        print(f"Epoch {epoch + 1}/{max_epochs} - Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1s}")
        print(f"Epoch {epoch + 1}/{max_epochs} - Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1s}")
        
        # Early stopping logic
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_train_acc = train_acc
            best_train_precisions = train_precisions
            best_train_recalls = train_recalls
            best_train_f1s = train_f1s
            best_val_acc = val_acc
            best_val_precisions = val_precisions
            best_val_recalls = val_recalls
            best_val_f1s = val_f1s
            torch.save(model.state_dict(), save_path)
            epochs_no_improve = 0
            print("Model saved!")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print("Early stopping triggered")
                break
    
    total_train_time = perf_counter() - start_train
    return (best_train_acc, best_train_precisions, best_train_recalls, best_train_f1s,
            best_val_acc, best_val_precisions, best_val_recalls, best_val_f1s, total_train_time)

In [15]:
def evaluate_model(model, test_dataloader):
    model.eval()
    predictions = []
    true_labels = []
    classification_times = []
    
    start_test = perf_counter()
    
    with torch.no_grad():
        for batch in tqdm(test_dataloader, desc="Testing"):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)  # Shape: (batch_size, num_classes)
            
            for i in range(input_ids.size(0)):
                input_id = input_ids[i].unsqueeze(0)
                attention_mask_sample = attention_mask[i].unsqueeze(0)
                label = labels[i].cpu().numpy()  # Shape: (num_classes,)
                
                start_time = perf_counter()
                
                output = model(input_ids=input_id, attention_mask=attention_mask_sample)  # Shape: (1, num_classes)
                pred = (torch.sigmoid(output) > 0.5).float().cpu().numpy()[0]  # Shape: (num_classes,)
                
                predictions.append(pred)
                true_labels.append(label)
                classification_times.append(perf_counter() - start_time)
    
    total_test_time = perf_counter() - start_test
    print(f"Test Time: {total_test_time:.2f} seconds")
    
    predictions = np.array(predictions)  # Shape: (num_samples, num_classes)
    true_labels = np.array(true_labels)  # Shape: (num_samples, num_classes)
    
    acc, precisions, recalls, f1s = get_metrics(true_labels, predictions)
    
    print("Test Metrics:")
    print("Accuracy:", acc)
    print("F1s:", f1s)
    print("Precisions:", precisions)
    print("Recalls:", recalls)
    
    return predictions, true_labels

In [ ]:
train_texts = train_df['text'].values
train_labels = train_df.drop(columns=['text']).values

val_texts = val_df['text'].values
val_labels = val_df.drop(columns=['text']).values

test_texts = test_df['text'].values
test_labels = test_df.drop(columns=['text']).values

num_classes = train_labels.shape[1]

# Create datasets
train_dataset = MultiLabelClassificationDataset(train_texts, train_labels, tokenizer, MAX_LEN)
val_dataset = MultiLabelClassificationDataset(val_texts, val_labels, tokenizer, MAX_LEN)
test_dataset = MultiLabelClassificationDataset(test_texts, test_labels, tokenizer, MAX_LEN)

seeds = [2, 3, 5]
results = []

for seed in seeds:
    torch.manual_seed(seed)
    model = BertForMultiLabelClassification(num_classes).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.BCEWithLogitsLoss()  # For multi-label classification

    train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
    test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

    save_path = f'results/bert_multilabel3_bs{BATCH_SIZE}_lr{LEARNING_RATE}_seed{seed}.pt'

    # Train
    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    max_memory_usage_train, retval = memory_usage(
        (train_model, (model, train_dataloader, val_dataloader, optimizer, criterion, save_path),
         {'max_epochs': MAX_EPOCHS, 'patience': PATIENCE}), max_usage=True, retval=True)

    max_vram_usage_train = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

    (train_acc, train_precisions, train_recalls, train_f1s,
     val_acc, val_precisions, val_recalls, val_f1s, total_train_time) = retval

    # Load best model
    model.load_state_dict(torch.load(save_path))

    # Evaluate
    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    start = perf_counter()
    max_memory_usage_test, test_retval = memory_usage(
        (evaluate_model, (model, test_dataloader), {}), max_usage=True, retval=True)
    total_time_test = perf_counter() - start

    max_vram_usage_test = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

    predictions, true_labels = test_retval
    test_acc, test_precisions, test_recalls, test_f1s = get_metrics(true_labels, predictions)

    # Store individual seed results
    results.append({
        'seed': seed,
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'train_acc': train_acc,
        'train_precisions': train_precisions.tolist(),
        'train_recalls': train_recalls.tolist(),
        'train_f1s': train_f1s.tolist(),
        'max_memory_usage_train': max_memory_usage_train,
        'max_vram_usage_train': max_vram_usage_train,
        'total_train_time': total_train_time,
        'val_acc': val_acc,
        'val_precisions': val_precisions.tolist(),
        'val_recalls': val_recalls.tolist(),
        'val_f1s': val_f1s.tolist(),
        'test_acc': test_acc,
        'test_precisions': test_precisions.tolist(),
        'test_recalls': test_recalls.tolist(),
        'test_f1s': test_f1s.tolist(),
        'max_memory_usage_test': max_memory_usage_test,
        'max_vram_usage_test': max_vram_usage_test,
        'total_test_time': total_time_test
    })

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(
Epoch 1/4:   0%|          | 0/3 [00:00<?, ?it/s]c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\transformers\models\bert\modeling_bert.py:440: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\transformers\models\bert\modeling_bert.py:440: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.funct

Epoch 1/4 - Train Loss: 0.6624, Acc: 0.1538, F1: [0.05714286 0.3030303  0.859375   0.20408163 0.20833333]
Epoch 1/4 - Val Loss: 0.6130, Acc: 0.5000, F1: [0.         0.         0.82352941 0.         0.        ]
Model saved!
Model saved!


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c

Epoch 2/4 - Train Loss: 0.5942, Acc: 0.4231, F1: [0.33333333 0.         0.88571429 0.         0.        ]
Epoch 2/4 - Val Loss: 0.5672, Acc: 0.5000, F1: [0.         0.         0.82352941 0.         0.        ]
Model saved!
Model saved!


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c

...

C:\Users\Rafael\AppData\Local\Temp\ipykernel_7540\357557947.py:46: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(save_path))
c:\Users\Rafael

Test Time: 0.09 seconds
Test Metrics:
Accuracy: 0.6
F1s: [0.         0.         0.94736842 0.         0.        ]
Precisions: [0.  0.  0.9 0.  0. ]
Recalls: [0. 0. 1. 0. 0.]


Testing: 100%|██████████| 1/1 [00:00<00:00, 11.41it/s]

Test Time: 0.09 seconds
Test Metrics:
Accuracy: 0.6
F1s: [0.         0.         0.94736842 0.         0.        ]
Precisions: [0.  0.  0.9 0.  0. ]
Recalls: [0. 0. 1. 0. 0.]



c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
df = pd.DataFrame(results)
df.to_csv('results/bert_multilabel3.csv', index=False)

In [18]:
df

,seed,batch_size,learning_rate,train_acc,train_precisions,train_recalls,train_f1s,max_memory_usage_train,max_vram_usage_train,total_train_time,...,val_precisions,val_recalls,val_f1s,test_acc,test_precisions,test_recalls,test_f1s,max_memory_usage_test,max_vram_usage_test,total_test_time
0,2,32,0.00002,0.461538,"[0.5, 0.0, 0.7948717948717948, 0.0, 0.0]","[0.037037037037037035, 0.0, 1.0, 0.0, 0.0]","[0.06896551724137931, 0.0, 0.8857142857142857,...",1320.917969,3806.438965,5.641418,...,"[0.0, 0.0, 0.7, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.8235294117647058, 0.0, 0.0]",0.6,"[0.0, 0.0, 0.9, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.9473684210526315, 0.0, 0.0]",1337.738281,1759.541016,1.303683
1,3,32,0.00002,0.461538,"[0.0, 0.0, 0.7948717948717948, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.8857142857142857, 0.0, 0.0]",1341.921875,3826.063965,5.376083,...,"[0.0, 0.0, 0.7, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.8235294117647058, 0.0, 0.0]",0.6,"[0.0, 0.0, 0.9, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.9473684210526315, 0.0, 0.0]",1341.921875,1769.166016,1.169487
2,5,32,0.00002,0.461538,"[0.0, 0.0, 0.7948717948717948, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.8857142857142857, 0.0, 0.0]",1342.285156,3816.438965,5.240686,...,"[0.0, 0.0, 0.7, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.8235294117647058, 0.0, 0.0]",0.6,"[0.0, 0.0, 0.9, 0.0, 0.0]","[0.0, 0.0, 1.0, 0.0, 0.0]","[0.0, 0.0, 0.9473684210526315, 0.0, 0.0]",1342.164062,1769.916016,1.178211
